# 03 EDA and time-series analysis (train period only)
Run after `python -m src.data.preprocessing`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.analysis import tsa
s_all = pd.read_csv(ROOT / "data/processed/fred_RSAFSNA.csv", index_col=0, parse_dates=True)["sales"]
train = s_all.iloc[: int(len(s_all) * 0.70)]   # analysis on TRAIN only (avoids test leakage into model choice)
print(len(train), train.index.min().date(), train.index.max().date())

In [ ]:
ax = train.plot(figsize=(12,4), title='Training series'); train.rolling(12).mean().plot(ax=ax); plt.show()

In [ ]:
dec = tsa.stl_decompose(train, 12); dec.plot(subplots=True, figsize=(12,8)); print(tsa.component_strength(dec)); plt.show()

In [ ]:
ap, band = tsa.acf_pacf(train, 48)
fig, ax = plt.subplots(1,2, figsize=(14,4)); ax[0].bar(ap.lag[1:], ap.acf[1:]); ax[1].bar(ap.lag[1:], ap.pacf[1:])
for a in ax: a.axhline(band, ls='--', c='r'); a.axhline(-band, ls='--', c='r')
ax[0].set_title('ACF'); ax[1].set_title('PACF'); plt.show()

In [ ]:
rows = []
for name, (d, D) in {'original': (0,0), 'd=1': (1,0), 'D=1': (0,1), 'd=1,D=1': (1,1)}.items():
    x = tsa.diff_series(train, d, D, 12); a, k = tsa.adf_test(x), tsa.kpss_test(x)
    rows.append((name, a['p'], k['p'], tsa.stationarity_verdict(a['p'], k['p'])))
pd.DataFrame(rows, columns=['series','ADF p','KPSS p','verdict'])

## Notes (write your own interpretation here from the measured output above)